Creation of a Temporary table for all the regions in Beijing via Database Query

## Import Libraries

Load pandas for data manipulation and SQLAlchemy for connecting to the MySQL database.

In [4]:
import pandas as pd
from sqlalchemy import create_engine

## Connect to DATABASE

Create and test a connection to the `air_quality` database.

In [16]:
engine = create_engine(
    "mysql+pymysql://root:@localhost/air_quality"
)

try:
    with engine.connect():
        print("Connected to air_quality!")
except Exception as error:
    print(f"Failed to connect to air_quality: {error}")

Connected to air_quality!


## Combine Station Tables

Load each station table, print its row count, and combine all station data into `df_combined`. The combined data is also exported to `regions.csv`.

In [15]:
table_names = [
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
]

dataframes = []
source_row_counts = {}

for table_name in table_names:
    dataframe = pd.read_sql_query(
        f"SELECT * FROM `{table_name}`",
        engine
    )
    dataframes.append(dataframe)
    source_row_counts[table_name] = len(dataframe)
    print(f"{table_name}: {len(dataframe):,} rows")

df_combined = pd.concat(dataframes, ignore_index=True)
df_combined.to_csv('regions.csv', index=False)

source_total_rows = sum(source_row_counts.values())
combined_total_rows = len(df_combined)

print(f"\nRows from all source tables: {source_total_rows:,}")
print(f"Rows in df_combined: {combined_total_rows:,}")
print(f"Row counts match: {source_total_rows == combined_total_rows}")

aotizhongxin: 35,064 rows
changping: 35,064 rows
dingling: 35,064 rows
dongsi: 35,064 rows
guanyuan: 35,064 rows
gucheng: 35,064 rows
huairou: 35,064 rows
nongzhanguan: 35,064 rows
shunyi: 35,064 rows
tiantan: 35,064 rows
wanliu: 35,064 rows
wanshouxigong: 35,064 rows

Rows from all source tables: 420,768
Rows in df_combined: 420,768
Row counts match: True


## Validate Stations

Compare the stations found in `df_combined` with the expected station list and report missing or unexpected stations.

## Check for All Stations

Compare the stations in `df_combined` with the expected list and report missing or unexpected stations.

In [12]:
expected_stations = {
    "aotizhongxin", "changping", "dingling", "dongsi",
    "guanyuan", "gucheng", "huairou", "nongzhanguan",
    "shunyi", "tiantan", "wanliu", "wanshouxigong"
}

if "station" not in df_combined.columns:
    raise KeyError("The 'station' column was not found in df_combined.")

present_stations = set(df_combined["station"].dropna().str.lower().unique())
missing_stations = expected_stations - present_stations
unexpected_stations = present_stations - expected_stations

print(f"Expected stations: {len(expected_stations)}")
print(f"Stations found: {len(present_stations)}")
print(f"Missing stations: {sorted(missing_stations) or 'None'}")
print(f"Unexpected stations: {sorted(unexpected_stations) or 'None'}")

if not missing_stations and not unexpected_stations:
    print("All expected stations are present.")
else:
    print("The station list does not exactly match the expected stations.")

Expected stations: 12
Stations found: 12
Missing stations: None
Unexpected stations: None
All expected stations are present.


## Load Combined Data into MySQL

Store `df_combined` in the `air_quality` database as the `all_regions` table.

## Load Combined Data into DATABASE

Store `df_combined` in the `air_quality` database 

In [13]:
df_combined.to_sql(
    "all_regions",
    con=engine,
    if_exists="replace",
    index=False,
    chunksize=1000
)

print(f"Loaded {len(df_combined):,} rows into the all_regions table.")

Loaded 420,768 rows into the all_regions table.
